# Phase 0: Machbarkeits-Check

Dieses Notebook prüft, ob das vortrainierte Modell `keremberke/yolov5m-clash-of-clans` für die CoC Base Analyzer Pipeline nutzbar ist.

**Ziele:**
1. Modell laden und Inferenz ausführen
2. 20–30 Screenshots aus `ml/tests/regression_set/th{level}/` testen
3. Pro Bild: korrekte Detektionen, False Positives, False Negatives manuell erfassen
4. Ergebnisse nach `phase0_results.csv` exportieren

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import pandas as pd
import matplotlib.pyplot as plt
import yolov5

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == 'ml':
    PROJECT_ROOT = PROJECT_ROOT.parent

ML_ROOT = PROJECT_ROOT / 'ml'
REGRESSION_SET = ML_ROOT / 'tests' / 'regression_set'
NOTEBOOKS_DIR = ML_ROOT / 'notebooks'
RESULTS_CSV = NOTEBOOKS_DIR / 'phase0_results.csv'

MODEL_ID = 'keremberke/yolov5m-clash-of-clans'
CLASS_NAMES = [
    'ad', 'airsweeper', 'bombtower', 'canon', 'clancastle', 'eagle', 'inferno',
    'kingpad', 'mortar', 'queenpad', 'rcpad', 'scattershot', 'th13', 'wardenpad',
    'wizztower', 'xbow',
]

print(f'Project root: {PROJECT_ROOT}')
print(f'Regression set: {REGRESSION_SET}')

## 1. Modell laden

In [ ]:
model = yolov5.load(MODEL_ID)
model.conf = 0.25
model.iou = 0.45
model.max_det = 1000
print(f'Model loaded: {MODEL_ID}')
print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')

## 2. Testbilder finden

Lege Screenshots unter `ml/tests/regression_set/th{10-18}/` ab (PNG/JPG).

In [ ]:
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp'}

def find_regression_images() -> list[Path]:
    images = []
    for p in sorted(REGRESSION_SET.rglob('*')):
        if p.suffix.lower() in IMAGE_EXTS and p.is_file():
            images.append(p)
    return images

def infer_th_level(path: Path) -> str:
    for part in path.parts:
        if part.startswith('th') and part[2:].isdigit():
            return part
    return 'unknown'

images = find_regression_images()
print(f'Found {len(images)} regression set image(s)')
for img in images[:10]:
    print(f'  {infer_th_level(img)}: {img.name}')
if len(images) > 10:
    print(f'  ... and {len(images) - 10} more')

### Fallback: HF-Dataset-Sample (nur Smoke Test)

Wenn keine lokalen Screenshots vorhanden sind, wird ein Bild aus dem Trainings-Dataset geladen.

In [ ]:
if not images:
    from datasets import load_dataset
    sample_dir = NOTEBOOKS_DIR / 'hf_sample_images'
    sample_dir.mkdir(parents=True, exist_ok=True)
    out = sample_dir / 'hf_test_000.jpg'
    if not out.exists():
        ds = load_dataset('keremberke/clash-of-clans-object-detection', name='full', split='test')
        ds[0]['image'].save(out)
    images = [out]
    print(f'Using HF sample: {out}')
else:
    print('Using local regression set images')

## 3. Inferenz auf allen Testbildern

In [ ]:
def run_inference(image_path: Path) -> dict:
    results = model(str(image_path), size=640)
    predictions = results.pred[0]
    if predictions is None or len(predictions) == 0:
        return {'count': 0, 'classes': [], 'scores': [], 'results': results}
    categories = predictions[:, 5].int().tolist()
    scores = predictions[:, 4].tolist()
    class_names = [CLASS_NAMES[int(c)] if int(c) < len(CLASS_NAMES) else str(int(c)) for c in categories]
    return {'count': len(predictions), 'classes': class_names, 'scores': [round(s, 3) for s in scores], 'results': results}

inference_rows = []
for img_path in images:
    det = run_inference(img_path)
    try:
        rel = str(img_path.relative_to(PROJECT_ROOT))
    except ValueError:
        rel = str(img_path)
    inference_rows.append({
        'image_path': rel,
        'th_level': infer_th_level(img_path),
        'model_detections': det['count'],
        'detected_classes': json.dumps(det['classes']),
        'detection_scores': json.dumps(det['scores']),
        'correct': '',
        'false_positives': '',
        'false_negatives': '',
        'notes': '',
        'evaluated_at': '',
        'run_timestamp': datetime.now(timezone.utc).isoformat(),
    })
    print(f"{img_path.name}: {det['count']} detections")

df = pd.DataFrame(inference_rows)
df

## 4. Visualisierung (Einzelbild)

Setze `IMAGE_INDEX` auf das zu prüfende Bild.

In [ ]:
IMAGE_INDEX = 0
img_path = images[IMAGE_INDEX]
det = run_inference(img_path)
det['results'].show()
print(f"Classes: {det['classes']}")
print(f"Scores: {det['scores']}")

## 5. Manuelle Evaluation

Für jedes Bild: zähle **correct**, **false_positives**, **false_negatives** und trage sie unten ein.

| Metrik | Bedeutung |
|--------|----------|
| correct | Modell-Box trifft tatsächliches Gebäude/Verteidigung |
| false_positives | Modell erkennt etwas, das nicht da ist / falsche Klasse |
| false_negatives | Sichtbares Gebäude, das das Modell nicht erkennt |

In [ ]:
# Manuelle Bewertungen: image_path -> {correct, false_positives, false_negatives, notes}
MANUAL_EVAL = {
    # Beispiel:
    # 'ml/tests/regression_set/th13/base_001.png': {
    #     'correct': 12,
    #     'false_positives': 2,
    #     'false_negatives': 5,
    #     'notes': 'TH13 base, eagle detected correctly',
    # },
}

for idx, row in df.iterrows():
    key = row['image_path']
    if key in MANUAL_EVAL:
        ev = MANUAL_EVAL[key]
        df.at[idx, 'correct'] = ev.get('correct', '')
        df.at[idx, 'false_positives'] = ev.get('false_positives', '')
        df.at[idx, 'false_negatives'] = ev.get('false_negatives', '')
        df.at[idx, 'notes'] = ev.get('notes', '')
        df.at[idx, 'evaluated_at'] = datetime.now(timezone.utc).isoformat()

df

## 6. Ergebnisse exportieren

In [ ]:
NOTEBOOKS_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(RESULTS_CSV, index=False)
print(f'Saved: {RESULTS_CSV}')
print(f'Rows: {len(df)}')

evaluated = df['correct'].astype(str).str.len() > 0
if evaluated.any():
    print('\nSummary (evaluated images):')
    sub = df[evaluated]
    print(f"  Total correct: {pd.to_numeric(sub['correct'], errors='coerce').sum():.0f}")
    print(f"  Total FP: {pd.to_numeric(sub['false_positives'], errors='coerce').sum():.0f}")
    print(f"  Total FN: {pd.to_numeric(sub['false_negatives'], errors='coerce').sum():.0f}")
else:
    print('\nNo manual evaluations yet — fill MANUAL_EVAL dict above.')